In [12]:
# Install dependencies
!pip install timm diffusers tqdm accelerate


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [18]:
# Install PyTorch with CUDA 12.8
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio #--index-url https://pypi.tuna.tsinghua.edu.cn/whl/cu128


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/49/8a/94bdecd13f5aaa90d45920b89789d9fe7c6f4af8c3cdd7ce01fcb59908fc/torch-2.12.0-cp313-cp313-manylinux_2_28_x86_64.whl (532.3 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/f6/24/4d0d48684251bd0673f87d633d5d88ab00227983b00591156eed2f86c8d5/torchvision-0.27.0-cp313-cp313-manylinux_2_28_x86_64.whl (7.6 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/4f/98/be13fe35d9aa5c26381c0e453c828a789d15c007f8f7d08c95341d19974d/torchaudio-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl (1.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision]━━━━━━ 2/3 [torchvision]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# can ignore this
!pip install flash-attn --no-build-isolation --no-cache-dir


In [19]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.12.0+cu130
CUDA: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


In [5]:
# IMPORTANT: This notebook is designed for YOUR CUSTOM DiT fork with x2 modifications.
# 
# Option 1: If you've pushed your code to GitHub, replace the URL below:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO
#
# Option 2: For now, we clone the base DiT repo and overwrite with your custom files:
%cd /home/ubuntu/
!git clone -b optimze-train-timing https://github.com/mrdjango/DuoDiT.git
%cd DuoDiT
!git checkout optimze-train-timing
!git pull

# The next cells will overwrite models.py (with x2 modifications) and add new scripts


/home/ubuntu
Cloning into 'DuoDiT'...
remote: Enumerating objects: 555, done.
remote: Counting objects: 100% (555/555), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 555 (delta 321), reused 543 (delta 312), pack-reused 0 (from 0)
Receiving objects: 100% (555/555), 31.09 MiB | 15.98 MiB/s, done.
Resolving deltas: 100% (321/321), done.
/home/ubuntu/DuoDiT
Already on 'optimze-train-timing'
Your branch is up to date with 'origin/optimze-train-timing'.
Already up to date.


In [6]:
# Download ImageNet Validation Dataset (ILSVRC2012)
# This will download ~6.3GB and organize it into class folders

import os
import tarfile
from pathlib import Path
import xml.etree.ElementTree as ET

# Download validation set
print('Downloading ImageNet validation set (~6.3GB)...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar

# Download validation ground truth annotations
print('Downloading validation ground truth...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz

# Extract validation images
print('Extracting validation images...')
os.makedirs('imagenet_val', exist_ok=True)
!tar -xf ILSVRC2012_img_val.tar -C imagenet_val

# Extract devkit to get ground truth
print('Extracting devkit...')
!tar -xzf ILSVRC2012_devkit_t12.tar.gz

# Parse ground truth from devkit
print('Parsing validation ground truth...')
gt_file = 'ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt'
with open(gt_file, 'r') as f:
    # Ground truth file has 1-indexed class IDs, convert to 0-indexed
    labels = [int(line.strip()) - 1 for line in f]

# Organize into class folders
print('Organizing into class folders...')
val_dir = Path('imagenet_val')
organized_dir = Path('imagenet_val_organized')

# Create class directories
for class_id in set(labels):
    (organized_dir / str(class_id)).mkdir(parents=True, exist_ok=True)

# Move images to class folders
val_images = sorted(val_dir.glob('ILSVRC2012_val_*.JPEG'))
for idx, img_path in enumerate(val_images):
    class_id = labels[idx]
    target_path = organized_dir / str(class_id) / img_path.name
    img_path.rename(target_path)

print(f'✓ ImageNet validation set organized into {len(set(labels))} class folders')
print(f'Total images: {len(val_images)}')
print('Dataset ready at: ./imagenet_val_organized')


--2026-05-26 15:23:57--  https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar
Resolving image-net.org (image-net.org)... 171.64.68.16
Connecting to image-net.org (image-net.org)|171.64.68.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6744924160 (6.3G) [application/x-tar]
Saving to: ‘ILSVRC2012_img_val.tar’

ILSVRC2012_img_val. 100%[===================>]   6.28G  18.7MB/s    in 6m 33s  

2026-05-26 15:30:31 (16.4 MB/s) - ‘ILSVRC2012_img_val.tar’ saved [6744924160/6744924160]

--2026-05-26 15:30:32--  https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz
Resolving image-net.org (image-net.org)... 171.64.68.16
Connecting to image-net.org (image-net.org)|171.64.68.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2568145 (2.4M) [application/x-gzip]
Saving to: ‘ILSVRC2012_devkit_t12.tar.gz’

ILSVRC2012_devkit_t 100%[===================>]   2.45M  2.54MB/s    in 1.0s    

2026-05-26 15:30:33 (2.54 MB/s) - ‘ILSVR

In [25]:
# Custom results directory
!torchrun --nnodes=1 --nproc_per_node=1 --master_addr=127.0.0.1 --master_port=29500 train_x2_finetune.py \
    --model DiT-XL/2 \
    --data-path ./imagenet_val_organized \
    --results-dir ./results/checkpoints \
    --pretrained-ckpt DiT-XL-2-256x256.pt \
    --epochs 400 \
    --classes $(python3 -c "print(' '.join(map(str, range(1000))))") \
    --global-batch-size 50 \
    --log-every 500 \
    --ckpt-every 500 \
    # --num-workers 2 \
    # --prefetch-factor 2 \
    # --debug-progress \
    # --progress-leave


Starting rank=0, seed=42, world_size=1.
[2026-05-28 13:28:31] Experiment directory created at ./results/checkpoints/003-DiT-XL-2-x2-finetune
Creating DiT-XL/2 model...
[DiT] Loading pre-trained ViT model: vit_large_patch16_224
[2026-05-28 13:28:38] Loading pretrained weights from Hugging Face hub (timm/vit_large_patch16_224.augreg_in21k_ft_in1k)
[2026-05-28 13:28:39] HTTP Request: HEAD https://huggingface.co/timm/vit_large_patch16_224.augreg_in21k_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[2026-05-28 13:28:39] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-05-28 13:28:39] [timm/vit_large_patch16_224.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
[DiT] Pre-trained ViT block: dim=1024, num_heads=16
[DiT] Target x2_vit_block: dim=1152, num_heads=16
[DiT] Dimensions don't match. Creating blo

In [18]:
!python3 -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

2.11.0+cu129 12.9 True NVIDIA H200


In [ ]:
import os
import shutil
import socket
import platform
import subprocess
import psutil
import torch

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

# Host / OS
print(f"Hostname: {socket.gethostname()}")
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")

print("\n" + "=" * 60)
print("CPU INFORMATION")
print("=" * 60)

print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU usage: {psutil.cpu_percent(interval=1)}%")

try:
    cpu_name = subprocess.check_output(
        "lscpu | grep 'Model name'",
        shell=True
    ).decode().strip()
    print(cpu_name)
except:
    pass

print("\n" + "=" * 60)
print("RAM INFORMATION")
print("=" * 60)

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Used RAM: {ram.used / (1024**3):.2f} GB")
print(f"RAM Usage: {ram.percent}%")

print("\n" + "=" * 60)
print("DISK INFORMATION")
print("=" * 60)

disk = shutil.disk_usage("/")

print(f"Total Disk: {disk.total / (1024**3):.2f} GB")
print(f"Used Disk: {disk.used / (1024**3):.2f} GB")
print(f"Free Disk: {disk.free / (1024**3):.2f} GB")

print("\n" + "=" * 60)
print("GPU INFORMATION")
print("=" * 60)

print(f"Torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"GPU count: {gpu_count}")

    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)

        total_mem = props.total_memory / (1024**3)

        allocated = torch.cuda.memory_allocated(i) / (1024**3)
        reserved = torch.cuda.memory_reserved(i) / (1024**3)

        print("\n" + "-" * 40)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"VRAM Total: {total_mem:.2f} GB")
        print(f"VRAM Allocated: {allocated:.2f} GB")
        print(f"VRAM Reserved: {reserved:.2f} GB")
        print(f"Compute Capability: {props.major}.{props.minor}")
        print(f"Multiprocessors: {props.multi_processor_count}")

print("\n" + "=" * 60)
print("NVIDIA-SMI")
print("=" * 60)

try:
    os.system("nvidia-smi")
except:
    print("nvidia-smi not available")